In [50]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [51]:
class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int

    sr: float
    bpb: float
    boundary_percent: float
    summary: str

In [ ]:
def calculate_sr(state: BatsmanState):
    sr = (state['runs']/state['balls']) * 100
    return {'sr': sr}

In [53]:
def calculate_bpb(state: BatsmanState):
    bpb = state['balls']/(state['fours'] + state['sixes'])
    return {'bpb' : bpb}

In [54]:
def calculate_boundary_percent(state: BatsmanState):
    boundary_percent = (((state['fours'] * 4)  + (state['sixes']*6 )) / state['runs']) * 100 
    return {'boundary_percent' : boundary_percent}

In [55]:
def summary(state: BatsmanState):
    summary = f"""
Strike Rate - {state['sr']} \n
Balls per boundary - {state['bpb']} \n
Boundary percent - {state['boundary_percent']}
"""
    state['summary'] = summary
    return state

In [56]:
graph = StateGraph(BatsmanState)

graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node('summary', summary)


In [57]:
graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_bpb')
graph.add_edge(START, 'calculate_boundary_percent')
graph.add_edge('calculate_sr', 'summary')
graph.add_edge('calculate_bpb', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')

graph.add_edge('summary', END)

workflow = graph.compile()


In [58]:
initial_state = {
    'runs': 120,
    'balls': 50,
    'fours': 6,
    'sixes': 4
}

workflow.invoke(initial_state)

{'runs': 120,
 'balls': 50,
 'fours': 6,
 'sixes': 4,
 'sr': 0.024,
 'bpb': 5.0,
 'boundary_percent': 40.0,
 'summary': '\nStrike Rate - 0.024 \n\nBalls per boundary - 5.0 \n\nBoundary percent - 40.0\n'}